# v3.4 on an A100 — the version without the ankle cut, on the failure set and the controls

Brand-new notebook for v3.4. **No ankle cut anywhere** (arm `Vnc`). Two matrices: the 31-pair v3.3 failure set and 30 clean controls, at **new seeds** (49/50/51) so the run answers the regression-to-the-mean question — does a fresh A100 draw rescue as many failing pairs as fal did? Reuses the iron-man run's inputs and A4 crops from Drive. Every call timed; cost in `meta/cost.json`.

Open directly: https://colab.research.google.com/github/101011101/magichour_takehome/blob/v3.3-lock/v3/colab/v34_a100.ipynb — nothing to upload.

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs (the unit is whatever you put here)
SEEDS = [49, 50, 51]          # NEW seeds - not the 46/47/48 of the iron-man run, on purpose
ARMS = ("Vnc",)               # the locked v3.3 version WITHOUT the ankle cut; nothing else changes
MATRICES = ["v34_failures.csv", "v34_controls.csv"]
DRIVE_PROJECT_DIR = "Side projects and shi"
PREV_RUN_ZIP = "v33_ironman_run_20260830_0548.zip"   # in Drive v3_runs/: its inputs and A4 crops are reused

In [ ]:
# 2 · Drive: find the HF cache that holds klein; reuse the iron-man run
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'], '(klein cached)' if found else '(no cached klein - cell 4 downloads ~13 GB once)')
PREV = os.path.join(BASE, 'v3_runs', PREV_RUN_ZIP); print('previous run:', PREV, os.path.exists(PREV))

In [ ]:
# 3 · install; pull the bundle from GitHub (public, branch v3.3-lock)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
!cd /content && rm -rf v34 && wget -q -O v33_ironman_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip && unzip -qo v33_ironman_bundle.zip -d v34
%cd /content/v34
import os, zipfile, onnxruntime as ort, torch
assert os.path.exists('lib/run_ironman.py') and os.path.exists('v34_failures.csv') and os.path.exists('v34_controls.csv'), 'bundle incomplete'
zipfile.ZipFile(PREV).extractall('run'); print('iron-man inputs + crops unpacked:', len(os.listdir('run/inputs')), 'files')
for f in os.listdir('run/gen'): os.remove('run/gen/' + f)          # a clean gen/ - this run's outputs only
print('onnxruntime providers:', ort.get_available_providers(), '| gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first
import run_ironman as R
R.main(MATRICES[0], 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
print(sorted(f for f in os.listdir('run/gen')))

In [ ]:
# 6 · both matrices, all seeds (resumable)
for m in MATRICES:
    R.main(m, 'testset', limit=None, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))

In [ ]:
# 7 · zip this run's references, outputs and meta to Drive
import shutil, time
name = f"v34_a100_nocut_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if '__Vnc' in f: z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'): z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True); shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip'))
print('->', os.path.join(BASE, 'v3_runs', name + '.zip'), ' then locally: python3 v3/build/v34_a100_page.py <zip>')

## Link D — fal's call-2 canvas, same seeds as link C

The deep dive found two code-level differences from fal, both from one line: we size call 2 to image 1 at ≤1.15 MP / floor 16, fal to area 1024² / floor 32, up or down. Above 4,300 tokens `compute_empirical_mu` switches sigma schedule (38 of 200 iron-man outputs crossed it); below ~4,000 tokens we render small persons on far fewer tokens than fal. Arm `Vfc` = `Vnc` with fal's canvas on call 2, **seeds 49/50/51 — the same as link C**, so the only difference between `Vnc` and `Vfc` cells is the canvas.

In [ ]:
# 8 · link D: Vfc on both matrices at the link-C seeds (references reused from Vnc: same prompt, same crop)
import shutil, glob, os, zipfile, time, json
for f in glob.glob('run/refs/*__Vnc.jpg'):                     # Vfc uses the identical reference; only call 2 changes
    dst = f.replace('__Vnc.jpg', '__Vfc.jpg')
    if not os.path.exists(dst): shutil.copy(f, dst)
for f in glob.glob('run/refs/*__Vnc_uncut.jpg'):
    dst = f.replace('__Vnc_uncut.jpg', '__Vfc_uncut.jpg')
    if not os.path.exists(dst): shutil.copy(f, dst)
import run_ironman as R
for m in MATRICES:
    R.main(m, 'testset', limit=None, seeds=SEEDS, arms=('Vfc',), gpu_usd_per_hour=A100_USD_PER_HOUR)
print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))
name = f"v34_a100_falcanvas_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/gen'):
        if '__Vfc__' in f: z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip')); print('->', os.path.join(BASE, 'v3_runs', name + '.zip'))